# Product Category Performance Analysis

This notebook analyzes sales, returns, and margin performance across product categories using the Sample Superstore dataset. The focus is on revenue contribution, profitability quality, seasonality, and identifying strong versus weak categories.


## Project OverviewThis notebook evaluates category-level retail performance using the Sample Superstore workbook already present in the repository. It combines sales, profit, returns, seasonality, and score-based category ranking to show which product categories are strong, weak, or operationally risky.## Learning Objectives- Compare category performance using multiple commercial metrics at once.- Distinguish between revenue leaders and true margin contributors.- Incorporate returns pressure and seasonality into category evaluation.- Translate category-level results into practical assortment and promotion decisions.## Business / Research ProblemRetail teams need to know which product categories truly deserve emphasis. High sales alone are not enough if margins are weak, return rates are high, or seasonal volatility makes execution difficult.## Why This Analysis MattersCategory-level analysis supports assortment planning, pricing strategy, inventory focus, and operational prioritisation. It helps teams invest attention where the commercial trade-offs are most meaningful.

## Dataset OverviewThis notebook uses the `Orders` and `Returns` sheets from the repo-local `Sample - Superstore.xls` workbook. Key fields include `Category`, `Sub-Category`, `Sales`, `Profit`, `Discount`, `Quantity`, `Order Date`, and return indicators merged from the returns sheet.## Dataset Source and License NotesThe workbook is a repo-local copy of the widely circulated Sample Superstore training dataset. Confirm the redistribution terms for the specific external source copy you use outside this repository. This notebook only analyzes the local workbook already available in the workspace.

## Environment SetupThe next cell installs the small set of packages needed to work with the Superstore workbook and produce the category analysis visuals.

In [1]:
import importlib, subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'xlrd']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment setup complete.')

Environment setup complete.


## Imports

## Exploratory Data AnalysisThe next cells establish the main category-performance questions and summarise the top-line retail context before drilling into deeper category-specific patterns.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


## Configuration / ConstantsThis block locates the Superstore workbook in the workspace so the analysis stays reproducible and does not depend on a hard-coded machine-specific path.

## Data Validation ChecksBefore category comparisons, we verify that both workbook sheets load correctly and that the expected retail columns are available for the merge and KPI calculations.

In [3]:
def locate_superstore_workbook() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd / 'Time Series Analysis' / 'Time Series Forecasting' / 'Sample - Superstore.xls')
    candidates.append(cwd.parent / 'Time Series Analysis' / 'Time Series Forecasting' / 'Sample - Superstore.xls')

    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / 'Time Series Analysis' / 'Time Series Forecasting' / 'Sample - Superstore.xls')

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError('Could not locate Sample - Superstore.xls from the current working directory.')

workbook_path = locate_superstore_workbook()
workbook_path


WindowsPath('E:/Github/Machine-Learning-Projects/Time Series Analysis/Time Series Forecasting/Sample - Superstore.xls')

In [4]:
orders = pd.read_excel(workbook_path, sheet_name='Orders')
returns = pd.read_excel(workbook_path, sheet_name='Returns')

print('Orders shape:', orders.shape)
print('Returns shape:', returns.shape)
orders.head()


Orders shape: (9994, 21)
Returns shape: (296, 2)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


In [5]:
returns['is_returned'] = returns['Returned'].astype(str).str.strip().str.lower().eq('yes')

data = orders.merge(returns[['Order ID', 'is_returned']], on='Order ID', how='left')
data['is_returned'] = data['is_returned'].fillna(False)

data['Order Date'] = pd.to_datetime(data['Order Date'])
data['Ship Date'] = pd.to_datetime(data['Ship Date'])
data['order_month'] = data['Order Date'].dt.to_period('M').dt.to_timestamp()
data['month_name'] = data['Order Date'].dt.month_name().str.slice(0, 3)
data['year'] = data['Order Date'].dt.year
data['profit_margin_pct'] = np.where(data['Sales'].ne(0), data['Profit'] / data['Sales'] * 100, np.nan)
data['returned_sales'] = np.where(data['is_returned'], data['Sales'], 0.0)
data['returned_profit'] = np.where(data['is_returned'], data['Profit'], 0.0)

data[['Category', 'Sub-Category', 'Sales', 'Profit', 'Discount', 'is_returned', 'order_month']].head()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\301934863.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['is_returned'] = data['is_returned'].fillna(False)


,Category,Sub-Category,Sales,Profit,Discount,is_returned,order_month
0,Furniture,Bookcases,261.96,41.91,0.00,False,2016-11-01
1,Furniture,Chairs,731.94,219.58,0.00,False,2016-11-01
2,Office Supplies,Labels,14.62,6.87,0.00,False,2016-06-01
3,Furniture,Tables,957.58,-383.03,0.45,False,2015-10-01
4,Office Supplies,Storage,22.37,2.52,0.20,False,2015-10-01


In [6]:
summary = pd.Series({
    'rows': len(data),
    'date_range_start': data['Order Date'].min(),
    'date_range_end': data['Order Date'].max(),
    'categories': data['Category'].nunique(),
    'subcategories': data['Sub-Category'].nunique(),
    'orders': data['Order ID'].nunique(),
    'returned_orders': int(data.loc[data['is_returned'], 'Order ID'].nunique())
})
summary


rows                               9994
date_range_start    2014-01-03 00:00:00
date_range_end      2017-12-30 00:00:00
categories                            3
subcategories                        17
orders                             5009
returned_orders                     296
dtype: object

## Univariate Analysis
The first pass is a category-level KPI table combining revenue, profit, margin, discount intensity, and return pressure. This becomes the base for the rest of the notebook.


In [7]:
category_kpis = (
    data.groupby('Category')
    .agg(
        sales=('Sales', 'sum'),
        profit=('Profit', 'sum'),
        quantity=('Quantity', 'sum'),
        orders=('Order ID', 'nunique'),
        avg_discount=('Discount', 'mean'),
        returned_sales=('returned_sales', 'sum'),
        returned_profit=('returned_profit', 'sum')
    )
    .sort_values('sales', ascending=False)
)
category_kpis['sales_share_pct'] = category_kpis['sales'] / category_kpis['sales'].sum() * 100
category_kpis['profit_share_pct'] = np.where(category_kpis['profit'].sum() != 0, category_kpis['profit'] / category_kpis['profit'].sum() * 100, np.nan)
category_kpis['profit_margin_pct'] = np.where(category_kpis['sales'].ne(0), category_kpis['profit'] / category_kpis['sales'] * 100, np.nan)
category_kpis['return_rate_pct'] = (
    data.groupby('Category')['is_returned'].mean() * 100
)
category_kpis['returned_sales_pct'] = np.where(category_kpis['sales'].ne(0), category_kpis['returned_sales'] / category_kpis['sales'] * 100, np.nan)
category_kpis = category_kpis.reset_index()
category_kpis


,Category,sales,profit,quantity,orders,avg_discount,returned_sales,returned_profit,sales_share_pct,profit_share_pct,profit_margin_pct,return_rate_pct,returned_sales_pct
0,Technology,"836,154.03","145,454.95",6939,1544,0.13,"72,708.17","13,997.35",36.40,50.79,17.40,8.45,8.70
1,Furniture,"741,999.80","18,451.27",8028,1764,0.17,"59,219.17","2,341.12",32.30,6.44,2.49,8.06,7.98
2,Office Supplies,"719,047.03","122,490.80",22906,3742,0.16,"48,576.93","6,893.90",31.30,42.77,17.04,7.85,6.76


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=category_kpis.sort_values('sales', ascending=False), x='sales', y='Category', palette='Blues_r', ax=axes[0])
axes[0].set_title('Revenue Contribution by Category')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('')

sns.barplot(data=category_kpis.sort_values('profit', ascending=False), x='profit', y='Category', palette='Greens_r', ax=axes[1])
axes[1].set_title('Profit Contribution by Category')
axes[1].set_xlabel('Profit')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\2132083449.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_kpis.sort_values('sales', ascending=False), x='sales', y='Category', palette='Blues_r', ax=axes[0])
C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\2132083449.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_kpis.sort_values('profit', ascending=False), x='profit', y='Category', palette='Greens_r', ax=axes[1])


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\2132083449.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Data CleaningThe notebook now standardises the return flag and merges the returns sheet into the orders table so category KPIs use one consistent analytic dataset.

## Visual AnalysisThe following charts compare category performance visually so strong and weak categories can be identified faster than from tabular KPIs alone.

In [9]:
fig, ax = plt.subplots(figsize=(10, 6))
margin_view = category_kpis.sort_values('profit_margin_pct', ascending=False)
sns.barplot(data=margin_view, x='profit_margin_pct', y='Category', palette='viridis', ax=ax)
ax.axvline(category_kpis['profit_margin_pct'].mean(), color='black', linestyle='--', linewidth=1.5, label='Category average')
ax.set_title('Profit Margin by Category')
ax.set_xlabel('Profit Margin (%)')
ax.set_ylabel('')
ax.legend()
plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1652961678.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=margin_view, x='profit_margin_pct', y='Category', palette='viridis', ax=ax)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1652961678.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Bivariate / Multivariate Analysis
Returns matter here at the order level. A category can look healthy on gross sales but still be weak if a large share of those sales comes back.


In [10]:
returns_view = category_kpis[['Category', 'sales', 'returned_sales', 'returned_sales_pct', 'return_rate_pct', 'profit']].sort_values('returned_sales_pct', ascending=False)
returns_view


,Category,sales,returned_sales,returned_sales_pct,return_rate_pct,profit
0,Technology,"836,154.03","72,708.17",8.70,8.45,"145,454.95"
1,Furniture,"741,999.80","59,219.17",7.98,8.06,"18,451.27"
2,Office Supplies,"719,047.03","48,576.93",6.76,7.85,"122,490.80"


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=returns_view, x='returned_sales_pct', y='Category', palette='Reds_r', ax=axes[0])
axes[0].set_title('Returned Sales as % of Category Revenue')
axes[0].set_xlabel('Returned Sales (%)')
axes[0].set_ylabel('')

sns.barplot(data=returns_view.sort_values('return_rate_pct', ascending=False), x='return_rate_pct', y='Category', palette='Oranges_r', ax=axes[1])
axes[1].set_title('Return Rate by Category')
axes[1].set_xlabel('Returned Orders (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1195621239.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=returns_view, x='returned_sales_pct', y='Category', palette='Reds_r', ax=axes[0])
C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1195621239.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=returns_view.sort_values('return_rate_pct', ascending=False), x='return_rate_pct', y='Category', palette='Oranges_r', ax=axes[1])
C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1195621239.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature-Specific Insights
To understand strong versus weak periods, the notebook tracks category revenue and profit month by month. This surfaces whether a category is consistently strong or only carried by a few seasonal spikes.


In [12]:
monthly_category = (
    data.groupby(['order_month', 'Category'])
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
    .reset_index()
)
monthly_category.head()


,order_month,Category,sales,profit
0,2014-01-01,Furniture,"6,242.53",805.47
1,2014-01-01,Office Supplies,"4,851.08",788.95
2,2014-01-01,Technology,"3,143.29",855.77
3,2014-02-01,Furniture,"1,839.66",120.69
4,2014-02-01,Office Supplies,"1,071.72",176.09


In [13]:
fig, ax = plt.subplots(figsize=(16, 7))
sns.lineplot(data=monthly_category, x='order_month', y='sales', hue='Category', marker='o', ax=ax)
ax.set_title('Monthly Revenue by Category')
ax.set_xlabel('Order Month')
ax.set_ylabel('Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\3777133804.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
monthly_heatmap = (
    data.assign(month_num=data['Order Date'].dt.month)
    .groupby(['Category', 'month_num'])['Sales']
    .sum()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(monthly_heatmap, cmap='YlGnBu', annot=True, fmt='.0f', linewidths=0.5, ax=ax)
ax.set_title('Seasonality Heatmap: Sales by Category and Month')
ax.set_xlabel('Month Number')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1243593625.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
monthly_margin = (
    data.groupby(['order_month', 'Category'])
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
    .reset_index()
)
monthly_margin['profit_margin_pct'] = np.where(monthly_margin['sales'].ne(0), monthly_margin['profit'] / monthly_margin['sales'] * 100, np.nan)

fig, ax = plt.subplots(figsize=(16, 7))
sns.lineplot(data=monthly_margin, x='order_month', y='profit_margin_pct', hue='Category', marker='o', ax=ax)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Monthly Profit Margin by Category')
ax.set_xlabel('Order Month')
ax.set_ylabel('Profit Margin (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\1105521801.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Strong vs Weak Category Scoring

A category is treated as stronger when it combines large revenue share, positive margin, and lower return drag. The score below is a transparent heuristic, not a statistical model.


In [16]:
score_table = category_kpis.copy()
score_table['revenue_score'] = score_table['sales_share_pct'].rank(pct=True)
score_table['margin_score'] = score_table['profit_margin_pct'].rank(pct=True)
score_table['returns_score'] = 1 - score_table['returned_sales_pct'].rank(pct=True)
score_table['overall_score'] = (
    0.45 * score_table['revenue_score'] +
    0.35 * score_table['margin_score'] +
    0.20 * score_table['returns_score']
)
score_table = score_table.sort_values('overall_score', ascending=False)
score_table[['Category', 'sales', 'profit', 'profit_margin_pct', 'returned_sales_pct', 'overall_score']]


,Category,sales,profit,profit_margin_pct,returned_sales_pct,overall_score
0,Technology,"836,154.03","145,454.95",17.40,8.70,0.80
2,Office Supplies,"719,047.03","122,490.80",17.04,6.76,0.52
1,Furniture,"741,999.80","18,451.27",2.49,7.98,0.48


## Statistical ChecksThese checks test whether categories differ materially on profit margin and return pressure rather than relying on visual ranking alone.

In [17]:
from scipy import stats
stat_df = category_kpis[['Category', 'profit_margin_pct', 'returned_sales_pct']].dropna()
margin_groups = [g['profit_margin_pct'].values for _, g in stat_df.groupby('Category') if len(g) > 0]
return_groups = [g['returned_sales_pct'].values for _, g in stat_df.groupby('Category') if len(g) > 0]
h_margin, p_margin = stats.kruskal(*margin_groups) if len(margin_groups) >= 2 else (None, None)
h_return, p_return = stats.kruskal(*return_groups) if len(return_groups) >= 2 else (None, None)
if h_margin is not None:
    print(f'Kruskal-Wallis test for category profit margin differences: H={h_margin:.3f}, p={p_margin:.2e}')
    print(f'Kruskal-Wallis test for category return-pressure differences: H={h_return:.3f}, p={p_return:.2e}')
else:
    print('Not enough groups for Kruskal-Wallis test')

Kruskal-Wallis test for category profit margin differences: H=2.000, p=3.68e-01
Kruskal-Wallis test for category return-pressure differences: H=2.000, p=3.68e-01


## Common Mistakes- Ranking categories on sales alone without checking profit or returns.- Ignoring seasonality when interpreting weak category months.- Treating returned sales as operational noise instead of a commercial signal.- Comparing categories without normalising for margin structure.- Making assortment decisions from one KPI instead of a balanced scorecard.

## Mini Challenge / Exercises1. Recompute the category score after doubling the weight on profit margin. Which category rises or falls most?2. Identify the sub-categories driving return pressure inside the weakest category.3. Compare holiday-season versus non-holiday-season category performance.4. Test whether high-discount categories systematically have weaker margins.5. Add shipping cost if available and rebuild the category scorecard.

In [18]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    category_kpis['sales_share_pct'],
    category_kpis['profit_margin_pct'],
    s=category_kpis['returned_sales_pct'].fillna(0) * 80 + 200,
    c=category_kpis['return_rate_pct'],
    cmap='coolwarm',
    alpha=0.8,
    edgecolor='black'
)

for _, row in category_kpis.iterrows():
    ax.text(row['sales_share_pct'] + 0.2, row['profit_margin_pct'] + 0.2, row['Category'], fontsize=11)

ax.set_title('Category Positioning: Revenue Share vs Margin')
ax.set_xlabel('Revenue Share (%)')
ax.set_ylabel('Profit Margin (%)')
colorbar = plt.colorbar(scatter)
colorbar.set_label('Return Rate (%)')
plt.tight_layout()
plt.show()


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\151600476.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Summary / Key TakeawaysCategory performance is multi-dimensional. The strongest categories are not just large on sales, but commercially balanced across margin, returns, and seasonal stability. This notebook keeps those dimensions together so category decisions stay practical and defensible.

In [19]:
weak_periods = (
    monthly_margin.sort_values(['Category', 'profit_margin_pct'])
    .groupby('Category')
    .head(3)
    .sort_values(['Category', 'profit_margin_pct'])
)
weak_periods[['Category', 'order_month', 'sales', 'profit', 'profit_margin_pct']]


,Category,order_month,sales,profit,profit_margin_pct
36,Furniture,2015-01-01,"11,739.94","-3,014.20",-25.67
135,Furniture,2017-10-01,"21,884.07","-2,526.92",-11.55
6,Furniture,2014-03-01,"14,573.96","-1,128.66",-7.74
19,Office Supplies,2014-07-01,"15,121.21","-2,482.02",-16.41
82,Office Supplies,2016-04-01,"10,647.45","-1,100.48",-10.34
142,Office Supplies,2017-12-01,"30,436.94","1,774.31",5.83
119,Technology,2017-04-01,"12,383.39","-2,639.77",-21.32
38,Technology,2015-01-01,"4,625.35",-856.70,-18.52
104,Technology,2016-11-01,"27,141.06","-1,495.33",-5.51


In [20]:
strong_categories = score_table.head(2)[['Category', 'overall_score', 'sales_share_pct', 'profit_margin_pct', 'returned_sales_pct']]
weak_categories = score_table.tail(2)[['Category', 'overall_score', 'sales_share_pct', 'profit_margin_pct', 'returned_sales_pct']]

print('Strong categories')
display(strong_categories)
print('Weak categories')
display(weak_categories)


Strong categories


,Category,overall_score,sales_share_pct,profit_margin_pct,returned_sales_pct
0,Technology,0.80,36.40,17.40,8.70
2,Office Supplies,0.52,31.30,17.04,6.76


Weak categories


,Category,overall_score,sales_share_pct,profit_margin_pct,returned_sales_pct
2,Office Supplies,0.52,31.30,17.04,6.76
1,Furniture,0.48,32.30,2.49,7.98


In [21]:
discount_impact = (
    data.groupby('Category')
    .apply(lambda frame: pd.Series({
        'avg_discount': frame['Discount'].mean(),
        'avg_margin_pct': np.where(frame['Sales'].sum() != 0, frame['Profit'].sum() / frame['Sales'].sum() * 100, np.nan),
        'corr_discount_profit': frame['Discount'].corr(frame['Profit'])
    }))
    .reset_index()
)
discount_impact


C:\Users\ahmad\AppData\Local\Temp\ipykernel_10632\2607876645.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda frame: pd.Series({


,Category,avg_discount,avg_margin_pct,corr_discount_profit
0,Furniture,0.17,2.4866951334588845,-0.48
1,Office Supplies,0.16,17.03515839002865,-0.21
2,Technology,0.13,17.395712076891932,-0.27


## Key Findings
Use the KPI and seasonality views together:

- High revenue plus healthy margin indicates scalable categories worth protecting in inventory and pricing.
- High revenue with weak margin often points to discount dependence or cost pressure.
- High return rates are a separate risk signal, especially when returned sales represent a material share of category revenue.
- Strong seasonality suggests timing inventory, promotions, and staffing to peak demand windows.


In [22]:
insight_table = category_kpis[['Category', 'sales_share_pct', 'profit_margin_pct', 'returned_sales_pct', 'return_rate_pct']].copy()
insight_table['revenue_tier'] = pd.qcut(insight_table['sales_share_pct'], q=3, labels=['Low', 'Medium', 'High'])
insight_table['margin_tier'] = pd.qcut(insight_table['profit_margin_pct'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])
insight_table['returns_tier'] = pd.qcut(insight_table['returned_sales_pct'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])
insight_table


,Category,sales_share_pct,profit_margin_pct,returned_sales_pct,return_rate_pct,revenue_tier,margin_tier,returns_tier
0,Technology,36.40,17.40,8.70,8.45,High,High,High
1,Furniture,32.30,2.49,7.98,8.06,Medium,Low,Medium
2,Office Supplies,31.30,17.04,6.76,7.85,Low,Medium,Low


## Recommendations / Next Steps
1. Protect the strongest category with disciplined discounting if it combines high revenue share and high margin.
2. Review the weakest category's return causes before pushing more top-line sales into it.
3. Use the weakest monthly periods table to target corrective actions by season rather than treating performance as uniform across the year.
4. If a category shows high sales but weak margin, inspect product mix and promotion strategy at the sub-category level next.


## Limitations

- Returns are measured at the order level from the workbook's `Returns` sheet, not at the individual line-item reason-code level.
- Margin is based on the provided `Profit` field and does not include downstream handling costs for returns.
- This notebook ranks categories with a heuristic score to support prioritization, not a causal inference model.
